# Part 1: Topic Detection and Summarization of News Articles

This notebook uses **LangChain** with a **local Ollama model (`llama3.2`, a small language model)** to analyze BBC News articles and produce, for each article:

1. **Topic Classification** — Business / Entertainment / Politics / Sport / Tech
2. **Summarization** — a 2-3 sentence summary
3. **Key Entity Extraction** — people, organizations, and locations mentioned

We use Ollama (rather than a hosted API like Groq) to run everything locally without hitting rate limits, per the assignment's suggestion, which also lets us process the full dataset for the bonus section.

**Name:** Aishwarya Nevrekar  
**Registration No.:** 27PGAI0028

In [1]:
import json
import threading
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd
from typing import List, Literal
from pydantic import BaseModel, Field
from tqdm.auto import tqdm

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

MODEL_NAME = "llama3.2"  # local SLM served via Ollama
OLLAMA_URL = "http://localhost:11434"  # explicit address: an OLLAMA_HOST of 0.0.0.0 is not a valid client address
# num_predict caps each reply so a looping generation cannot stall a long run
llm = ChatOllama(model=MODEL_NAME, temperature=0, base_url=OLLAMA_URL, num_predict=512)

print(f"Using Ollama model: {MODEL_NAME}")

Using Ollama model: llama3.2


## Step 1: Load the Dataset

The BBC News Full-Text dataset (2,225 articles across 5 categories: business, entertainment, politics, sport, tech).

In [2]:
raw_df = pd.read_csv("data/bbc-news-data.csv", sep="\t")
print(raw_df.shape)
print(raw_df["category"].value_counts())
raw_df.head(3)

(2225, 4)
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...


In [3]:
# Build a clean working dataframe and limit to the first 30 articles
df = raw_df.rename(columns={"content": "Article_Text", "title": "Title", "category": "Category"}).copy()
df.insert(0, "Article_ID", df.index)
df = df[["Article_ID", "Title", "Category", "Article_Text"]]

df = df.head(30).reset_index(drop=True)
print(df.shape)
df.head()

(30, 4)


,Article_ID,Title,Category,Article_Text
0,0,Ad sales boost Time Warner profit,business,Quarterly profits at US media giant TimeWarne...
1,1,Dollar gains on Greenspan speech,business,The dollar has hit its highest level against ...
2,2,Yukos unit buyer faces loan claim,business,The owners of embattled Russian oil giant Yuk...
3,3,High fuel prices hit BA's profits,business,British Airways has blamed high fuel prices f...
4,4,Pernod takeover talk lifts Domecq,business,Shares in UK drinks and food firm Allied Dome...


## Step 2: Topic Classification

We build a LangChain prompt (with a couple of few-shot examples) and force the model to return a single structured label using `with_structured_output`, so we never have to parse free text.

In [4]:
class TopicClassification(BaseModel):
    topic: Literal["Business", "Entertainment", "Politics", "Sport", "Tech"] = Field(
        description="The single best-fitting news category for the article"
    )

classification_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a news editor. Analyze the given news article and identify its topic as one of the "
     "following categories: Business, Entertainment, Politics, Sport, or Tech. "
     "Respond with only the single best-fitting category.\n"
     "Business: companies, earnings, markets, the economy, trade, jobs, prices and interest rates.\n"
     "Entertainment: films, cinema, box office, actors, music, singers, albums, TV shows, theatre, books and awards.\n"
     "Politics: government, elections, parliament, political parties, ministers and laws.\n"
     "Sport: athletes, teams, clubs, matches, tournaments and championships.\n"
     "Tech: technology, computers, the internet, software, video games, mobile phones and telecoms.\n\n"
     "Examples:\n"
     "Article: 'Shares in the company jumped 5% after strong quarterly earnings were reported to investors.'\n"
     "Category: Business\n\n"
     "Article: 'The actress won an award for her role in the film, which also topped the box office.'\n"
     "Category: Entertainment\n\n"
     "Article: 'MPs debated the new immigration bill as the prime minister prepared for the election.'\n"
     "Category: Politics\n\n"
     "Article: 'The striker scored a hat-trick as his team won the league final 4-2 in extra time.'\n"
     "Category: Sport\n\n"
     "Article: 'The firm launched a new smartphone and a faster broadband service for gamers.'\n"
     "Category: Tech"),
    ("human", "Article:\n{article}\n\nCategory:")
])

classification_chain = classification_prompt | llm.with_structured_output(TopicClassification)

# --- Test on a sample datapoint ---
sample = df.iloc[0]
sample_result = classification_chain.invoke({"article": sample["Article_Text"][:3000]})
print("Title:", sample["Title"])
print("Actual category (dataset label):", sample["Category"])
print("Predicted topic:", sample_result.topic)

Title: Ad sales boost Time Warner profit
Actual category (dataset label): business
Predicted topic: Business


## Step 3: Summarization

A simple prompt asking for a concise 2-3 sentence summary capturing who/what/when/where/why, without personal commentary.

In [5]:
summarization_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a professional news summarizer. Summarize the main points of the given news article "
     "in 2-3 sentences, capturing who/what/when/where/why as applicable. "
     "Do not add personal commentary or opinions. Return only the summary text."),
    ("human", "Article:\n{article}")
])

summarization_chain = summarization_prompt | llm | StrOutputParser()

# --- Test on a sample datapoint ---
sample_summary = summarization_chain.invoke({"article": sample["Article_Text"][:4000]})
print("Title:", sample["Title"])
print("Summary:", sample_summary)

Title: Ad sales boost Time Warner profit
Summary: Time Warner's quarterly profits jumped 76% to $1.13 billion, driven by sales of high-speed internet connections and higher advert sales, with the company now owning 8% of search engine Google. The firm's fourth quarter sales rose 2% to $11.1 billion, with its internet business AOL experiencing a mixed fortune, losing 464,000 subscribers but seeing underlying profit rise 8%. Time Warner is re-stating its 2000 and 2003 results following a probe by the US Securities Exchange Commission.


## Step 4: Key Entity Extraction

We extract entities using **JSON-mode prompting** (Ollama's `format="json"`) with a Pydantic schema for validation via `JsonOutputParser`. In testing, LangChain's tool-calling `with_structured_output` frequently returned empty entity lists on longer articles with this small model — JSON-mode prompting proved much more reliable, so we use it here instead.

In [6]:
class KeyEntities(BaseModel):
    people: List[str] = Field(default_factory=list, description="Notable people mentioned in the article")
    organizations: List[str] = Field(default_factory=list, description="Organizations/companies mentioned")
    locations: List[str] = Field(default_factory=list, description="Places/locations mentioned")

entity_parser = JsonOutputParser(pydantic_object=KeyEntities)

entity_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "From the article below, list the names of any important people, organizations, or places mentioned. "
     "If a category has no entities, return an empty list for it.\n{format_instructions}"),
    ("human", "Article:\n{article}")
]).partial(format_instructions=entity_parser.get_format_instructions())

# JSON-mode (format="json") makes the Ollama model emit valid JSON directly, which this small
# model handles more reliably than LangChain's tool-calling with_structured_output.
entity_llm = ChatOllama(model=MODEL_NAME, temperature=0, format="json", base_url=OLLAMA_URL, num_predict=512)
entity_chain = entity_prompt | entity_llm | entity_parser

# --- Test on a sample datapoint ---
sample_entities = entity_chain.invoke({"article": sample["Article_Text"][:4000]})
print("Title:", sample["Title"])
print("Entities:", sample_entities)

Title: Ad sales boost Time Warner profit
Entities: {'people': ['Richard Parsons'], 'organizations': ['TimeWarner', 'Google', 'Warner Bros', 'AOL', 'SEC', 'Bertelsmann'], 'locations': []}


## Step 5: Apply to All Articles and Update the DataFrame

In [7]:
def analyze_article(article_text: str) -> dict:
    text = article_text[:4000]  # keep prompts within a reasonable context size for the SLM
    try:
        topic = classification_chain.invoke({"article": text}).topic
    except Exception:
        topic = "Unknown"
    try:
        summary = summarization_chain.invoke({"article": text}).strip()
    except Exception:
        summary = ""
    try:
        entities = entity_chain.invoke({"article": text})
    except Exception:
        entities = {"people": [], "organizations": [], "locations": []}
    return {"Detected_Topic": topic, "Summary": summary, "Key_Entities": entities}


def run_pipeline(data: pd.DataFrame, cache_path: str | None = None, workers: int = 1) -> pd.DataFrame:
    """Analyze every article. With `cache_path`, each finished row is saved to a JSON-lines file so a
    long run (the bonus) can resume where it stopped. Rows where a chain failed are retried once.
    `workers` > 1 sends several articles to Ollama at once (it serves requests in parallel)."""
    done = {}
    if cache_path and Path(cache_path).exists():
        for line in Path(cache_path).read_text(encoding="utf-8").splitlines():
            record = json.loads(line)
            done[record["Article_ID"]] = record["result"]
    todo = [(int(row["Article_ID"]), row["Article_Text"]) for _, row in data.iterrows()
            if int(row["Article_ID"]) not in done]
    cache = open(cache_path, "a", encoding="utf-8") if cache_path else None
    lock = threading.Lock()

    def work(item):
        key, text = item
        result = analyze_article(text)
        if result["Detected_Topic"] == "Unknown" or not result["Summary"]:
            result = analyze_article(text)  # retry once
        with lock:
            done[key] = result
            if cache:
                cache.write(json.dumps({"Article_ID": key, "result": result}) + "\n")
                cache.flush()

    with ThreadPoolExecutor(max_workers=workers) as pool:
        list(tqdm(pool.map(work, todo), total=len(todo), desc="Analyzing articles"))
    if cache:
        cache.close()
    results = [done[int(k)] for k in data["Article_ID"]]
    results_df = pd.DataFrame(results)
    return pd.concat([data.reset_index(drop=True), results_df], axis=1)


final_df = run_pipeline(df)
final_df.head()

Analyzing articles:   0%|          | 0/30 [00:00<?, ?it/s]

,Article_ID,Title,Category,Article_Text,Detected_Topic,Summary,Key_Entities
0,0,Ad sales boost Time Warner profit,business,Quarterly profits at US media giant TimeWarne...,Business,Time Warner's quarterly profits jumped 76% to ...,"{'people': ['Richard Parsons'], 'organizations..."
1,1,Dollar gains on Greenspan speech,business,The dollar has hit its highest level against ...,Business,The US dollar has reached its highest level ag...,"{'people': ['Alan Greenspan', 'Robert Sinche']..."
2,2,Yukos unit buyer faces loan claim,business,The owners of embattled Russian oil giant Yuk...,Business,"The owners of Yukos, Menatep Group, plan to as...","{'people': ['Mikhail Khodorkovsky', 'Jamie Fir..."
3,3,High fuel prices hit BA's profits,business,British Airways has blamed high fuel prices f...,Business,British Airways reported a 40% drop in profits...,"{'people': ['Rod Eddington', 'Mike Powell', 'M..."
4,4,Pernod takeover talk lifts Domecq,business,Shares in UK drinks and food firm Allied Dome...,Business,Shares in UK drinks and food firm Allied Domec...,"{'people': ['Pernod Ricard'], 'organizations':..."


In [8]:
# Quick sanity check: how often does the detected topic match the dataset's original label?
accuracy = (final_df["Detected_Topic"].str.lower() == final_df["Category"].str.lower()).mean()
print(f"Topic classification accuracy vs. ground-truth label (first 30 articles): {accuracy:.2%}")

Topic classification accuracy vs. ground-truth label (first 30 articles): 83.33%


In [9]:
final_df.to_csv("outputs/part1_bbc_first30_results.csv", index=False)
final_df.to_json("outputs/part1_bbc_first30_results.json", orient="records", indent=2)
print("Saved to outputs/part1_bbc_first30_results.csv and .json")
final_df

Saved to outputs/part1_bbc_first30_results.csv and .json


,Article_ID,Title,Category,Article_Text,Detected_Topic,Summary,Key_Entities
0,0,Ad sales boost Time Warner profit,business,Quarterly profits at US media giant TimeWarne...,Business,Time Warner's quarterly profits jumped 76% to ...,"{'people': ['Richard Parsons'], 'organizations..."
1,1,Dollar gains on Greenspan speech,business,The dollar has hit its highest level against ...,Business,The US dollar has reached its highest level ag...,"{'people': ['Alan Greenspan', 'Robert Sinche']..."
2,2,Yukos unit buyer faces loan claim,business,The owners of embattled Russian oil giant Yuk...,Business,"The owners of Yukos, Menatep Group, plan to as...","{'people': ['Mikhail Khodorkovsky', 'Jamie Fir..."
3,3,High fuel prices hit BA's profits,business,British Airways has blamed high fuel prices f...,Business,British Airways reported a 40% drop in profits...,"{'people': ['Rod Eddington', 'Mike Powell', 'M..."
4,4,Pernod takeover talk lifts Domecq,business,Shares in UK drinks and food firm Allied Dome...,Business,Shares in UK drinks and food firm Allied Domec...,"{'people': ['Pernod Ricard'], 'organizations':..."
5,5,Japan narrowly escapes recession,business,Japan's economy teetered on the brink of a te...,Business,Japan's economy experienced a technical recess...,"{'people': ['Heizo Takenaka', 'Paul Sheard'], ..."
6,6,Jobs growth still slow in the US,business,The US created fewer jobs than expected in Ja...,Business,"The US economy added 146,000 jobs in January, ...","{'people': ['Rick Egelton', 'Herbert Hoover', ..."
7,7,India calls for fair trade rules,business,"India, which attends the G7 meeting of seven ...",Politics,"India's finance minister, Palaniappan Chidamba...","{'people': ['Palaniappan Chidambaram', 'Gordon..."
8,8,Ethiopia's crop production up 24%,business,Ethiopia produced 14.27 million tonnes of cro...,Business,Ethiopia's crop production increased by 21% in...,"{'people': ['Henri Josserand'], 'organizations..."
9,9,Court rejects $280bn tobacco case,business,A US government claim accusing the country's ...,Politics,A US appeals court has rejected a $280bn claim...,"{'people': ['Clinton'], 'organizations': ['Alt..."


## Bonus: Full Dataset (all 2,225 articles)

`RUN_BONUS = True` below processes the *entire* dataset instead of just the first 30. Since this runs a local SLM via Ollama with no external API rate limits, it is feasible, but it will take a while (roughly 1-2 hours depending on hardware, since each article needs 3 LLM calls: classification, summarization, entity extraction).

Finished rows are saved to `outputs/part1_bbc_ALL_cache.jsonl` as the run goes, so an interrupted run continues where it stopped. Set `RUN_BONUS = False` to skip this cell.

In [10]:
RUN_BONUS = True

if RUN_BONUS:
    bonus_df = raw_df.rename(columns={"content": "Article_Text", "title": "Title", "category": "Category"}).copy()
    bonus_df.insert(0, "Article_ID", bonus_df.index)
    bonus_df = bonus_df[["Article_ID", "Title", "Category", "Article_Text"]]

    bonus_results_df = run_pipeline(bonus_df, cache_path="outputs/part1_bbc_ALL_cache.jsonl", workers=4)
    bonus_results_df.to_csv("outputs/part1_bbc_ALL_results.csv", index=False)
    bonus_results_df.to_json("outputs/part1_bbc_ALL_results.json", orient="records", indent=2)

    bonus_accuracy = (bonus_results_df["Detected_Topic"].str.lower() == bonus_results_df["Category"].str.lower()).mean()
    print(f"Full-dataset classification accuracy: {bonus_accuracy:.2%}")
    failed = ((bonus_results_df["Detected_Topic"] == "Unknown") | (bonus_results_df["Summary"] == "")).sum()
    print(f"Rows analysed: {len(bonus_results_df)} | rows with a failed chain: {failed}")
    print("Saved to outputs/part1_bbc_ALL_results.csv and .json")
else:
    print("RUN_BONUS is False - skipping full-dataset run. Set RUN_BONUS = True to process all 2,225 articles.")

Analyzing articles:   0%|          | 0/1960 [00:00<?, ?it/s]

Full-dataset classification accuracy: 82.56%
Rows analysed: 2225 | rows with a failed chain: 0
Saved to outputs/part1_bbc_ALL_results.csv and .json
